# 06 - Ensemble Weight Optimization

Learn blend weights with Grid Search, Bayesian Optimization, and FLAML.


## Fixed vs Optimized Weights

**Definition**  
Fixed weights assume equal model reliability; optimized weights adapt to empirical performance.

**Theory**  
Optimization minimizes validation/test loss over simplex-constrained weights.

**Mathematical Intuition**  
Constraint: `w_i >= 0` and `Σ w_i = 1`.

**Financial Intuition**  
Helps overweight models that stay calibrated in current market regime.

**Business Impact**  
Directly improves ensemble error and interpretability of model contribution.

**Real-World Example**  
Bayesian search can find better blends faster than exhaustive grids in high-dimensional spaces.

**Visual Explanation**  
Code cells below generate real plots from Apple market data.

**Code Explanation**  
Each code block is annotated inline and uses shared production modules from `src/`.

**Interpretation of Results**  
After each output, interpret what signal quality, risk, and forecasting reliability imply.


In [ ]:

import sys
import os
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = (PROJECT_ROOT / '..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.forecast_pipeline import ForecastingFramework
from src.data_loader import load_stock_data, split_data
from src.features import create_features
from src.evaluation import regression_metrics
from src.visualization import *

OUT = PROJECT_ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT_ROOT / 'config' / 'config.yaml'
FAST_NOTEBOOK_MODE = os.getenv('NOTEBOOK_FULL_RUN', '0') != '1'

def make_framework():
    framework = ForecastingFramework(str(CONFIG_PATH))
    if FAST_NOTEBOOK_MODE:
        framework.config['models']['automl']['lazypredict'] = False
        framework.config['models']['automl']['pycaret'] = False
        framework.config['models']['automl']['flaml'] = False
        framework.config['models']['deep_learning']['epochs'] = 8
        framework.config['models']['deep_learning']['early_stopping_patience'] = 3
        framework.config['weight_optimization']['methods'] = ['grid']
    return framework

framework = make_framework()


In [ ]:

framework = make_framework()
framework.load_data()
hybrid_out = framework.train_hybrids(horizon=1)

val_preds = hybrid_out['val_predictions']
y_val = hybrid_out['y_val_true']
test_preds = hybrid_out['test_predictions']
y_test = hybrid_out['y_test_true']

rows = []
for method in ['grid', 'bayesian', 'flaml']:
    try:
        result = framework.optimize_weights(
            1,
            val_preds,
            y_val,
            method=method,
            evaluation_predictions=test_preds,
            evaluation_y_true=y_test,
        )
        rows.append({
            'method': method,
            'val_rmse': result['fit_metrics']['rmse'],
            'test_rmse': result['test_metrics']['rmse'],
            'test_mae': result['test_metrics']['mae'],
            'test_mape': result['test_metrics']['mape'],
            'weights': result['weights'],
        })
    except Exception as exc:
        rows.append({'method': method, 'error': str(exc)})

opt_df = pd.DataFrame(rows)
display(opt_df)
opt_df.to_csv(OUT / 'tables/06_weight_optimization_h1.csv', index=False)
